In [5]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
else:
    print("No GPU available. Training will run on CPU.")

GPU: Tesla T4 is available.


# NetGuard Benchmark on Colab

In [37]:
# ================================
# 🚀 NETGUARD FULL PIPELINE (COLAB)
# ================================

import os
import sys
import subprocess
import datetime

# -------------------------------
# 🔧 CONFIG
# -------------------------------
dataset_to_run = "unsw_nb15"
max_samples = 100000
epochs = 10

models = [
    "bilstm_attention", "cnn_lstm", "contrastive_ssl",
    "vanilla_ae", "cnn1d", "ft_transformer",
    "isolation_forest", "vae"
]

# -------------------------------
# 📦 INSTALL DEPENDENCIES
# -------------------------------
!pip install -q torch pytorch-lightning scikit-learn pandas numpy plotly shap xgboost kaggle

# -------------------------------
# 🔑 KAGGLE AUTH (EDIT THIS)
# -------------------------------
os.environ['KAGGLE_USERNAME'] = 'saurabhkr'
os.environ['KAGGLE_KEY'] = 'KGAT_eda598a3dd5fe9e00d7041a4cb90d0'

# -------------------------------
# 📂 CLONE REPO
# -------------------------------
!cd /content && (git clone https://github.com/20Saurabh/Network-Anomaly-Detection.git 2>/dev/null || (cd Network-Anomaly-Detection && git pull))

repo_path = "/content/Network-Anomaly-Detection/backend"

if not os.path.exists(repo_path):
    raise Exception("❌ Repo not found!")

os.chdir(repo_path)
sys.path.insert(0, repo_path)

# -------------------------------
# 📥 DOWNLOAD DATASET
# -------------------------------
data_path = f"{repo_path}/data/raw/unsw_nb15"
os.makedirs(data_path, exist_ok=True)

print("📥 Downloading UNSW-NB15 dataset...")

!kaggle datasets download -d mrwellsdavid/unsw-nb15 -p {data_path}
!unzip -o {data_path}/unsw-nb15.zip -d {data_path}

# Fix structure if needed
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".csv"):
            src = os.path.join(root, file)
            dst = os.path.join(data_path, file)
            if src != dst:
                os.rename(src, dst)

print("✅ Dataset ready!")

# -------------------------------
# 🚀 RUN BENCHMARK
# -------------------------------
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"\n🚀 Starting Benchmark: {dataset_to_run} (ID: {run_id})")

for model in models:
    print(f"\n--- Training {model} ---")

    cmd = f"""
    python -u run_experiments.py \
    --dataset {dataset_to_run} \
    --epochs {epochs} \
    --runs 1 \
    --max-samples {max_samples} \
    --models {model}
    """

    result = subprocess.run(cmd, shell=True, text=True)

    if result.returncode != 0:
        print(f"❌ ERROR in {model}")
        continue

print("\n✅ ALL MODELS COMPLETED")

# -------------------------------
# 💾 SAVE RESULTS TO DRIVE
# -------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

drive_output = f'/content/drive/MyDrive/NetGuard_Results/runs/{run_id}'
!mkdir -p "{drive_output}"
!cp -r results/* "{drive_output}/"

print(f"\n📁 Saved to: {drive_output}")

Mounted at /content/drive

📁 Saved to: /content/drive/MyDrive/NetGuard_Results/runs/20260406_113540


## Generate Report

In [38]:
import json
import pandas as pd
import plotly.express as px
import os

# Load results
results_path = '/content/drive/MyDrive/NetGuard_Results/metrics/all_results.json'
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    # Convert to DataFrame (assuming results is a dict with model keys)
    df = pd.DataFrame.from_dict(results, orient='index').reset_index().rename(columns={'index': 'model'})
    
    # Display summary
    print("Benchmark Results Summary:")
    print(df.describe())
    
    # Plot performance comparison
    fig = px.bar(df, x='model', y='accuracy', title='Model Accuracy Comparison')
    fig.show()
    
    # Save report
    df.to_csv('/content/drive/MyDrive/NetGuard_Results/benchmark_report.csv', index=False)
    print("Report saved to /content/drive/MyDrive/NetGuard_Results/benchmark_report.csv")
else:
    print("Results not found. Run the benchmark first.")

Benchmark Results Summary:
       num_test_samples  num_classes  accuracy  precision    recall  f1_score  \
count               1.0          1.0  1.000000        1.0  1.000000  1.000000   
mean              750.0          2.0  0.973333        1.0  0.733333  0.846154   
std                 NaN          NaN       NaN        NaN       NaN       NaN   
min               750.0          2.0  0.973333        1.0  0.733333  0.846154   
25%               750.0          2.0  0.973333        1.0  0.733333  0.846154   
50%               750.0          2.0  0.973333        1.0  0.733333  0.846154   
75%               750.0          2.0  0.973333        1.0  0.733333  0.846154   
max               750.0          2.0  0.973333        1.0  0.733333  0.846154   

       f1_macro  f1_weighted  auc_roc    auc_pr  detection_rate  \
count  1.000000     1.000000  1.00000  1.000000        1.000000   
mean   0.915778     0.971477  0.99996  0.999647        0.733333   
std         NaN          NaN      NaN     

Report saved to /content/drive/MyDrive/NetGuard_Results/benchmark_report.csv
